# 配套实践 15-01：从部分观测形成 belief 并长程想象

本练习模拟一个只能观测位置、看不到速度的受控系统。单帧模型根据当前位置和动作预测下一位置；GRU belief 模型还把历史写入隐藏状态。训练后先用 6 步真实历史更新 belief，再关闭未来观测，让两个模型沿同一候选动作自由 rollout。依赖：PyTorch、NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/15-long-horizon-imagination-and-uncertainty/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 汇总训练曲线和 horizon 误差
import torch  # 生成部分观测序列并训练 belief 模型
from torch import nn  # 使用 GRUCell、多层感知机和回归头
import matplotlib.pyplot as plt  # 绘制隐藏速度歧义、训练表现和想象轨迹
torch.set_num_threads(2)  # 限制轻量实验的 CPU 线程开销
torch.manual_seed(151)  # 固定模型初始化与训练批次顺序
np.random.seed(151)  # 固定 NumPy 侧随机过程
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 相同位置观测可能包含不同隐藏速度

系统真实状态由位置和速度组成，但相机只返回位置。下面让三个系统从同一位置出发，初始速度分别向左、静止和向右，在前几步执行相同零动作。单帧观测无法判断下一步属于哪条轨迹。

In [ ]:
def advance_system(position, velocity, action):  # 定义具有隐藏速度的真实离散动力学
    next_velocity = 0.94 * velocity + 0.22 * action  # 根据阻尼和动作更新不可观测速度
    next_position = position + 0.25 * next_velocity  # 使用更新后速度推进可观测位置
    return next_position, next_velocity  # 返回完整真实状态供数据生成使用
initial_velocities = [-0.7, 0.0, 0.7]  # 设置向左、静止和向右三种隐藏速度
fig, axis = plt.subplots(figsize=(8.5, 3.8))  # 创建相同初始观测的未来分叉图
for initial_velocity, color in zip(initial_velocities, ["#2563eb", "#94a3b8", "#ea580c"]):  # 依次模拟三种隐藏速度
    position = 0.0  # 让三条轨迹拥有相同初始位置观测
    velocity = initial_velocity  # 设置当前轨迹不可直接观测的初始速度
    position_trace = [position]  # 保存该系统的可见位置历史
    for step_index in range(8):  # 在相同零动作下推进八步
        position, velocity = advance_system(position, velocity, 0.0)  # 使用相同动作更新真实状态
        position_trace.append(position)  # 保存当前可见位置
    axis.plot(position_trace, marker="o", color=color, label=f"Hidden velocity {initial_velocity:+.1f}")  # 显示隐藏速度造成的不同未来
axis.scatter([0], [0], color="#172033", s=90, zorder=4, label="Same observation")  # 强调三条轨迹共享初始观测
axis.set(title="One position observation cannot reveal hidden velocity", xlabel="Time step", ylabel="Observed position")  # 标注部分可观测问题
axis.legend()  # 显示三种隐藏速度图例
axis.grid(alpha=0.2)  # 添加淡网格帮助观察运动方向
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示相同当前观测对应的不同未来

**怎样理解结果：** 三条轨迹在第 0 步的可见位置完全相同，随后却朝不同方向发展。速度没有出现在单帧观测中，但连续位置变化可以让 recurrent belief 推断它。若历史只有一帧，任何确定性模型都只能在这三种未来之间平均。

## 2. 训练单帧模型与 recurrent belief

每条训练序列具有随机初始位置、隐藏速度和动作。单帧 MLP 每一步只读取当前位置与动作；GRUCell 还把此前位置—动作对压入 24 维 hidden belief。两者都用真实观测进行 teacher forcing 训练。

In [ ]:
def make_sequences(sequence_count, sequence_length, random_seed):  # 定义生成部分可观测控制序列的函数
    generator = torch.Generator().manual_seed(random_seed)  # 为当前数据集建立独立随机生成器
    positions = torch.empty(sequence_count).uniform_(-1.0, 1.0, generator=generator)  # 随机采样初始可见位置
    velocities = torch.empty(sequence_count).uniform_(-0.8, 0.8, generator=generator)  # 随机采样不可见初始速度
    position_steps = [positions]  # 保存包括初始位置在内的观测序列
    action_steps = []  # 保存每个转移对应的动作
    for step_index in range(sequence_length):  # 逐步产生随机动作与真实状态转移
        actions = torch.empty(sequence_count).uniform_(-1.0, 1.0, generator=generator)  # 为当前时间采样控制动作
        positions, velocities = advance_system(positions, velocities, actions)  # 用隐藏速度动力学推进所有序列
        action_steps.append(actions)  # 保存当前动作标签
        position_steps.append(positions)  # 保存动作后的下一位置观测
    return torch.stack(position_steps, dim=1), torch.stack(action_steps, dim=1)  # 返回位置序列和对齐动作序列
sequence_length = 30  # 设置每条训练序列包含三十个转移
train_positions, train_actions = make_sequences(1400, sequence_length, 31)  # 生成用于参数更新的序列
test_positions, test_actions = make_sequences(300, sequence_length, 32)  # 生成独立测试序列
class BeliefModel(nn.Module):  # 定义用 recurrent hidden state 维护 belief 的模型
    def __init__(self):  # 初始化 GRUCell 和位置残差头
        super().__init__()  # 初始化 PyTorch 模型基类
        self.cell = nn.GRUCell(2, 24)  # 使用当前位置与动作更新二十四维 belief
        self.delta_head = nn.Linear(24, 1)  # 从 belief 预测下一位置相对当前的变化
    def forward(self, position_sequence, action_sequence):  # 定义 teacher forcing 下的完整序列预测
        hidden = torch.zeros(len(position_sequence), 24)  # 为每条序列初始化空 belief
        predictions = []  # 准备保存各时间位置的下一观测预测
        for step_index in range(action_sequence.shape[1]):  # 按时间顺序更新 belief
            model_input = torch.stack([position_sequence[:, step_index], action_sequence[:, step_index]], dim=1)  # 组织当前真实位置与对齐动作
            hidden = self.cell(model_input, hidden)  # 把当前证据写入 recurrent belief
            next_position = position_sequence[:, step_index] + self.delta_head(hidden).squeeze(1)  # 根据 belief 预测下一位置残差
            predictions.append(next_position)  # 保存当前时间的预测
        return torch.stack(predictions, dim=1)  # 返回样本乘时间的下一位置预测
class SingleFrameModel(nn.Module):  # 定义不保存历史的单帧预测基线
    def __init__(self):  # 初始化小型位置残差网络
        super().__init__()  # 初始化 PyTorch 模型基类
        self.network = nn.Sequential(nn.Linear(2, 32), nn.Tanh(), nn.Linear(32, 1))  # 把当前位置和动作映射到下一位置残差
    def forward(self, position_sequence, action_sequence):  # 定义整段序列的独立逐帧预测
        model_inputs = torch.stack([position_sequence[:, :-1], action_sequence], dim=-1)  # 组织全部时间的当前位置与动作
        return position_sequence[:, :-1] + self.network(model_inputs).squeeze(-1)  # 独立预测每个下一位置而不传递记忆
belief_model = BeliefModel()  # 创建 recurrent belief World Model
single_frame_model = SingleFrameModel()  # 创建只看单帧的 World Model 基线
belief_optimizer = torch.optim.Adam(belief_model.parameters(), lr=0.004)  # 为 belief 模型建立优化器
single_optimizer = torch.optim.Adam(single_frame_model.parameters(), lr=0.004)  # 为单帧模型建立优化器
belief_test_history = []  # 保存 belief 模型每轮测试 MSE
single_test_history = []  # 保存单帧模型每轮测试 MSE
for epoch_index in range(50):  # 重复五十轮小批量序列训练
    shuffled_indices = torch.randperm(len(train_positions))  # 每轮打乱训练序列顺序
    for start_index in range(0, len(train_positions), 100):  # 每批使用一百条完整序列
        batch_indices = shuffled_indices[start_index:start_index + 100]  # 取出当前序列批次
        belief_predictions = belief_model(train_positions[batch_indices, :-1], train_actions[batch_indices])  # 使用真实历史更新 belief 并预测下一位置
        belief_loss = ((belief_predictions - train_positions[batch_indices, 1:]) ** 2).mean()  # 计算 belief 序列预测 MSE
        belief_optimizer.zero_grad()  # 清除 belief 模型上一批次梯度
        belief_loss.backward()  # 通过时间反向传播 belief 预测误差
        belief_optimizer.step()  # 更新 recurrent belief 参数
        single_predictions = single_frame_model(train_positions[batch_indices], train_actions[batch_indices])  # 独立预测当前批次每个下一位置
        single_loss = ((single_predictions - train_positions[batch_indices, 1:]) ** 2).mean()  # 计算单帧模型序列平均 MSE
        single_optimizer.zero_grad()  # 清除单帧模型上一批次梯度
        single_loss.backward()  # 反向传播单帧预测误差
        single_optimizer.step()  # 更新单帧模型参数
    with torch.no_grad():  # 关闭每轮测试过程的梯度记录
        belief_test_loss = ((belief_model(test_positions[:, :-1], test_actions) - test_positions[:, 1:]) ** 2).mean().item()  # 计算 belief 未见序列误差
        single_test_loss = ((single_frame_model(test_positions, test_actions) - test_positions[:, 1:]) ** 2).mean().item()  # 计算单帧未见序列误差
    belief_test_history.append(belief_test_loss)  # 保存当前轮 belief 测试误差
    single_test_history.append(single_test_loss)  # 保存当前轮单帧测试误差
fig, axis = plt.subplots(figsize=(8.8, 3.8))  # 创建两种部分观测模型的测试曲线
axis.semilogy(np.arange(1, 51), single_test_history, color="#94a3b8", label="Single frame")  # 绘制不含历史 belief 的测试误差
axis.semilogy(np.arange(1, 51), belief_test_history, color="#2563eb", label="GRU belief")  # 绘制 recurrent belief 的测试误差
axis.set(title="History helps infer the hidden velocity", xlabel="Epoch", ylabel="One-step test MSE (log scale)")  # 标注历史记忆的作用
axis.legend()  # 显示单帧与 belief 图例
axis.grid(alpha=0.2)  # 添加淡网格帮助比较数量级
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示部分可观测序列的训练结果

**怎样理解结果：** 单帧模型的误差停在较高平台，因为当前位置和动作不能唯一决定下一位置；GRU belief 通过连续位置变化推断隐藏速度，因此误差更低。它并没有获得真实速度标签，而是被下一位置预测目标迫使隐藏状态保存速度相关信息。

## 3. 先观测修正，再关闭观测进行 imagination

选择一条未见序列。前 6 步把真实位置和动作送入 GRU 更新 belief；随后只提供未来候选动作，模型把自己的预测位置重新作为下一步输入。单帧基线从第 6 步的真实位置开始同样自由 rollout。

In [ ]:
context_steps = 6  # 设置用于形成初始 belief 的真实历史长度
example_positions = test_positions[7]  # 选择一条固定未见位置序列
example_actions = test_actions[7]  # 读取与该序列严格对齐的动作
with torch.no_grad():  # 关闭 belief 更新和未来想象过程的梯度记录
    hidden = torch.zeros(1, 24)  # 初始化该测试序列的空 recurrent belief
    for step_index in range(context_steps):  # 使用前六步真实观测执行 posterior 式历史更新
        observed_input = torch.tensor([[example_positions[step_index].item(), example_actions[step_index].item()]])  # 组织当前真实位置与动作
        hidden = belief_model.cell(observed_input, hidden)  # 把真实历史证据写入 belief
    belief_position = example_positions[context_steps].reshape(1)  # 从最后真实观测位置开始未来想象
    single_position = example_positions[context_steps].reshape(1)  # 为单帧模型复制相同想象起点
    belief_imagination = [belief_position.item()]  # 保存 GRU belief 的无观测未来
    single_imagination = [single_position.item()]  # 保存单帧模型的无观测未来
    for step_index in range(context_steps, sequence_length):  # 沿剩余候选动作自由 rollout
        action_value = example_actions[step_index].reshape(1)  # 读取当前未来候选动作
        belief_input = torch.stack([belief_position, action_value], dim=1)  # 组织预测位置与动作供 belief 更新
        hidden = belief_model.cell(belief_input, hidden)  # 在没有真实未来观测时沿 prior 式路径更新 hidden
        belief_position = belief_position + belief_model.delta_head(hidden).squeeze(1)  # 使用 belief 预测并推进下一位置
        single_input = torch.stack([single_position, action_value], dim=1)  # 组织单帧模型自己的预测位置与动作
        single_position = single_position + single_frame_model.network(single_input).squeeze(1)  # 使用单帧残差模型自由推进
        belief_imagination.append(belief_position.item())  # 保存当前 belief 想象位置
        single_imagination.append(single_position.item())  # 保存当前单帧想象位置
true_future = example_positions[context_steps:].numpy()  # 取出同一时段的真实未来用于评价
belief_imagination = np.array(belief_imagination)  # 转换 GRU belief 未来为 NumPy 数组
single_imagination = np.array(single_imagination)  # 转换单帧未来为 NumPy 数组
horizons = np.arange(len(true_future))  # 建立从零开始的想象 horizon 横轴
belief_errors = np.abs(belief_imagination - true_future)  # 计算 belief 在每个 horizon 的绝对误差
single_errors = np.abs(single_imagination - true_future)  # 计算单帧模型每个 horizon 的绝对误差
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))  # 创建想象轨迹与 horizon 误差两幅图
axes[0].plot(horizons, true_future, color="#172033", linewidth=2.5, label="True future")  # 绘制真实未来位置
axes[0].plot(horizons, belief_imagination, color="#2563eb", label="GRU belief imagination")  # 绘制带历史 belief 的未来想象
axes[0].plot(horizons, single_imagination, color="#94a3b8", linestyle="--", label="Single-frame imagination")  # 绘制单帧自由 rollout
axes[0].set(title="Observed history ends at horizon 0", xlabel="Imagine horizon", ylabel="Position")  # 标注观测与想象边界
axes[0].legend()  # 显示真实和两种模型轨迹图例
axes[1].plot(horizons, belief_errors, color="#2563eb", label="GRU belief error")  # 绘制 belief 误差随 horizon 的变化
axes[1].plot(horizons, single_errors, color="#94a3b8", linestyle="--", label="Single-frame error")  # 绘制单帧误差随 horizon 的变化
axes[1].set(title="Small one-step errors compound during free rollout", xlabel="Imagine horizon", ylabel="Absolute error")  # 标注长程误差累积
axes[1].legend()  # 显示两种误差曲线图例
for axis in axes:  # 为两幅未来图统一添加网格
    axis.grid(alpha=0.2)  # 使用淡网格帮助读取 horizon 变化
fig.tight_layout()  # 调整两幅图间距
plt.show()  # 显示 belief 与单帧模型的长程想象差异

**怎样理解结果：** horizon 0 是最后一个真实观测，此后模型都不能再读取真实位置。GRU belief 根据前六步历史推断了隐藏速度，因此早期未来更接近真实轨迹；单帧模型缺少该信息，很快偏离。即使 belief 单步误差较小，自由 rollout 的误差仍会随 horizon 扩大。

**本练习的结论：** belief 解决“当前观测没有包含全部状态”的问题，不能消除 model bias。真实系统应在新观测到来时用 posterior 修正 belief，并在无观测 imagination 中同时传播不确定性。